In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import hashlib
import numpy as np
import shutil
import random
from PIL import Image
from tqdm import tqdm
import tensorflow as tf
import tensorflow.keras.backend as K
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time
import psutil, os
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report




load and preprocessing

In [ ]:
# @title
# load dataset

DATASET_DIR = "/content/drive/MyDrive/Projects/p11 - 121198 Manoj/Dataset/Chest X-ray Dataset"
CLASSES = ["yes", "no"]
# Remove corrupted / unreadable images
def remove_corrupted_images(dataset_dir):
    removed = 0
    for cls in CLASSES:
        cls_path = os.path.join(dataset_dir, cls)
        for img in tqdm(os.listdir(cls_path), desc=f"Checking {cls}"):
            img_path = os.path.join(cls_path, img)
            try:
                Image.open(img_path).verify()
            except:
                os.remove(img_path)
                removed += 1

    print(f"✅ Removed {removed} corrupted images")

remove_corrupted_images(DATASET_DIR)

# Remove duplicate images
def remove_duplicates(dataset_dir):
    hashes = {}
    removed = 0

    for cls in CLASSES:
        cls_path = os.path.join(dataset_dir, cls)
        for img in tqdm(os.listdir(cls_path), desc=f"Deduplicating {cls}"):
            img_path = os.path.join(cls_path, img)

            with open(img_path, "rb") as f:
                file_hash = hashlib.md5(f.read()).hexdigest()

            if file_hash in hashes:
                os.remove(img_path)
                removed += 1
            else:
                hashes[file_hash] = img_path

    print(f"✅ Removed {removed} duplicate images")
    # Verify labels
def verify_labels(dataset_dir):
    print("\n📊 Dataset summary:")
    for cls in CLASSES:
        cls_path = os.path.join(dataset_dir, cls)
        print(f"{cls}: {len(os.listdir(cls_path))} images")
verify_labels(DATASET_DIR)

# Convert all images to JPG
def convert_to_jpg(dataset_dir):
    converted = 0
    for cls in CLASSES:
        cls_path = os.path.join(dataset_dir, cls)
        for img in tqdm(os.listdir(cls_path), desc=f"Converting {cls}"):
            img_path = os.path.join(cls_path, img)
            try:
                image = Image.open(img_path).convert("RGB")
                new_path = img_path.rsplit(".", 1)[0] + ".jpg"
                image.save(new_path, "JPEG")
                if img_path != new_path:
                    os.remove(img_path)
                converted += 1
            except:
                pass
    print(f"✅ Converted {converted} images to JPG")
convert_to_jpg(DATASET_DIR)

# Standardize size & orientation
def standardize_images(dataset_dir, size=(50,50)):
    for cls in CLASSES:
        cls_path = os.path.join(dataset_dir, cls)
        for img in tqdm(os.listdir(cls_path), desc=f"Resizing {cls}"):
            img_path = os.path.join(cls_path, img)
            image = cv2.imread(img_path)
            if image is None:
                continue
            image = cv2.resize(image, size)
            cv2.imwrite(img_path, image)
    print("✅ Images resized to 50x50")
standardize_images(DATASET_DIR)

# Remove blank / empty slices
def remove_blank_images(dataset_dir, threshold=10):
    removed = 0
    for cls in CLASSES:
        cls_path = os.path.join(dataset_dir, cls)
        for img in tqdm(os.listdir(cls_path), desc=f"Removing blanks {cls}"):
            img_path = os.path.join(cls_path, img)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            if np.mean(image) < threshold:
                os.remove(img_path)
                removed += 1

    print(f"✅ Removed {removed} blank images")

remove_blank_images(DATASET_DIR)

# Noise Reduction
def apply_noise_reduction(dataset_dir):
    for label in ["yes", "no"]:
        class_path = os.path.join(dataset_dir, label)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            # Read image
            img = cv2.imread(img_path)
            if img is None:
                continue

            # Convert to grayscale (optional but useful)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Apply Bilateral Filter
            denoised = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

            # Save processed image (overwrite or save to new folder)
            cv2.imwrite(img_path, denoised)

    print("Noise reduction completed successfully.")
apply_noise_reduction(DATASET_DIR)


# Contrast Enhancement
# ------------------------------------
# Contrast Enhancement Function
# ------------------------------------
def apply_contrast_enhancement(dataset_dir):
    for label in ["yes", "no"]:
        class_path = os.path.join(dataset_dir, label)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            # Read image in grayscale
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # Create CLAHE object
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

            # Apply contrast enhancement
            enhanced = clahe.apply(img)

            # Save enhanced image
            cv2.imwrite(img_path, enhanced)

    print("✅ Contrast enhancement completed successfully.")

apply_contrast_enhancement(DATASET_DIR)

# Intensity Normalization
# ------------------------------------
# Intensity Normalization Function
# ------------------------------------
def apply_intensity_normalization(dataset_dir):
    for label in ["yes", "no"]:
        class_path = os.path.join(dataset_dir, label)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            # Read image in grayscale
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # Convert to float
            img = img.astype(np.float32)

            # Min-Max Normalization to [0, 1]
            min_val = np.min(img)
            max_val = np.max(img)

            if max_val > min_val:
                normalized = (img - min_val) / (max_val - min_val)
            else:
                normalized = img  # avoid division by zero

            # Convert back to 0–255 for saving
            normalized = (normalized * 255).astype(np.uint8)

            # Save normalized image
            cv2.imwrite(img_path, normalized)

    print("✅ Intensity normalization completed successfully.")
apply_intensity_normalization(DATASET_DIR)






Checking no: 100%|██████████| 100/100 [00:00<00:00, 202.26it/s]


✅ Removed 0 corrupted images

📊 Dataset summary:
yes: 96 images
no: 100 images


Converting no: 100%|██████████| 100/100 [00:01<00:00, 65.54it/s]


✅ Converted 196 images to JPG


Resizing no: 100%|██████████| 100/100 [00:01<00:00, 57.04it/s]


✅ Images resized to 50x50


Removing blanks no: 100%|██████████| 100/100 [00:00<00:00, 136.30it/s]


✅ Removed 0 blank images
Noise reduction completed successfully.
✅ Contrast enhancement completed successfully.
✅ Intensity normalization completed successfully.


Split dataset (Train & Test)

In [ ]:
# @title
# -----------------------------
# CONFIG
# -----------------------------
SOURCE_DIR = DATASET_DIR
DEST_DIR   = "/content/drive/MyDrive/Projects/p11 - 121198 Manoj/Dataset/Chest X-ray Dataset/split_50_50"

CLASSES = ["yes", "no"]
TRAIN_RATIO = 0.5
SEED = 42

random.seed(SEED)

# -----------------------------
# CREATE DIRECTORY STRUCTURE
# -----------------------------
for split in ["train", "test"]:
    for cls in CLASSES:
        os.makedirs(os.path.join(DEST_DIR, split, cls), exist_ok=True)

# -----------------------------
# SPLIT FUNCTION
# -----------------------------
def split_dataset():
    for cls in CLASSES:
        class_path = os.path.join(SOURCE_DIR, cls)
        images = os.listdir(class_path)
        random.shuffle(images)

        split_idx = int(len(images) * TRAIN_RATIO)

        train_imgs = images[:split_idx]
        test_imgs  = images[split_idx:]

        # Copy train images
        for img in train_imgs:
            src = os.path.join(class_path, img)
            dst = os.path.join(DEST_DIR, "train", cls, img)
            shutil.copy(src, dst)

        # Copy test images
        for img in test_imgs:
            src = os.path.join(class_path, img)
            dst = os.path.join(DEST_DIR, "test", cls, img)
            shutil.copy(src, dst)

        print(f"✅ {cls}: {len(train_imgs)} train | {len(test_imgs)} test")

split_dataset()

print("\n🎉 Dataset split completed successfully!")


✅ yes: 48 train | 48 test
✅ no: 50 train | 50 test

🎉 Dataset split completed successfully!


Data Augmentation

In [ ]:
# @title
IMG_SIZE = (50, 50)
BATCH_SIZE = 16
SEED = 42

# -----------------------------
# Load TRAIN dataset
# -----------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Projects/p11 - 121198 Manoj/Dataset/Chest X-ray Dataset/split_50_50/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED
)

# -----------------------------
# Load TEST dataset
# -----------------------------
test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Projects/p11 - 121198 Manoj/Dataset/Chest X-ray Dataset/split_50_50/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

# -----------------------------
# Data Augmentation (TRAIN ONLY)
# -----------------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

# Apply augmentation ONLY to train dataset
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

# Performance optimization
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.prefetch(tf.data.AUTOTUNE)


Found 98 files belonging to 2 classes.
Found 98 files belonging to 2 classes.


In [ ]:
for images, labels in train_ds.take(1):
    print(images.shape, images.dtype)
    print(labels.shape)

(16, 50, 50, 3) <dtype: 'float32'>
(16, 1)


ImTranNet-TriCore

In [ ]:
# @title
# ==================================================
# IMPORTS
# ==================================================
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import time, os

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ==================================================
# CUSTOM BLOCKS
# ==================================================
class LMAdaFilter(layers.Layer):
    def __init__(self):
        super().__init__()
        self.dw3 = layers.DepthwiseConv2D(3, padding="same")
        self.dw5 = layers.DepthwiseConv2D(5, padding="same")
        self.w = self.add_weight(shape=(2,), initializer="ones", trainable=True)
        self.pw = layers.Conv2D(16, 1)
        self.norm = layers.LayerNormalization()
        self.act = layers.ReLU()

    def call(self, x):
        f1 = self.dw3(x)
        f2 = self.dw5(x)
        w = tf.nn.softmax(self.w)
        out = w[0]*f1 + w[1]*f2
        out = self.pw(out)
        out = self.norm(out)
        return self.act(out + x)


class DP_AtRes_SRU_FAST(layers.Layer):
    def __init__(self, c):
        super().__init__()
        self.conv = layers.Conv2D(c, 3, padding="same", activation="relu")
        self.sru = layers.Conv1D(c, 3, padding="same", activation="relu")
        self.attn = layers.MultiHeadAttention(2, c//2)
        self.fuse = layers.Conv2D(c, 1)
        self.norm = layers.LayerNormalization()

    def call(self, x):
        B, H, W, C = tf.unstack(tf.shape(x))
        cnn = self.conv(x)
        seq = tf.reshape(x, [B, H*W, C])
        seq = self.attn(self.sru(seq), self.sru(seq))
        seq = tf.reshape(seq, [B, H, W, C])
        out = self.fuse(tf.concat([cnn, seq], axis=-1))
        out = self.norm(out)
        return tf.nn.relu(out + x)


class MHHT(layers.Layer):
    def __init__(self, c):
        super().__init__()
        self.attn = layers.MultiHeadAttention(1, c)
        self.ffn = models.Sequential([
            layers.Dense(32, activation="relu"),
            layers.Dense(c)
        ])
        self.n1 = layers.LayerNormalization()
        self.n2 = layers.LayerNormalization()

    def call(self, x):
        B, H, W, C = tf.unstack(tf.shape(x))
        seq = tf.reshape(x, [B, H*W, C])
        seq = self.n1(seq + self.attn(seq, seq))
        seq = self.n2(seq + self.ffn(seq))
        return tf.reshape(seq, [B, H, W, C])

# ==================================================
# MODEL
# ==================================================
def build_model():
    inp = layers.Input((50,50,3))
    x = layers.Conv2D(16, 3, padding="same", activation="relu")(inp)
    x = LMAdaFilter()(x)
    x = DP_AtRes_SRU_FAST(16)(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = MHHT(32)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1)(x)  # logits
    return models.Model(inp, out)

# ==================================================
# DATA
# ==================================================
TRAIN_DIR = "/content/drive/MyDrive/Projects/p11 - 121198 Manoj/Dataset/Chest X-ray Dataset/split_50_50/train"
TEST_DIR  = "/content/drive/MyDrive/Projects/p11 - 121198 Manoj/Dataset/Chest X-ray Dataset/split_50_50/test"

IMG_SIZE = (50,50)
BATCH = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH, label_mode="binary"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH, label_mode="binary"
)

norm = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)

# ==================================================
# CLASS WEIGHTS
# ==================================================
labels=[]
for _,y in train_ds:
    labels.extend(y.numpy().astype(int))

neg, pos = labels.count(0), labels.count(1)
class_weight = {
    0:(neg+pos)/(2*neg),
    1:(neg+pos)/(2*pos)
}

# ==================================================
# TRAIN
# ==================================================
model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(3e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.0)]
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    class_weight=class_weight,
    steps_per_epoch=2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# ==================================================
# EVALUATION (PROBABILITIES)
# ==================================================
y_true, y_prob = [], []

for x,y in test_ds:
    logits = model.predict(x, verbose=0)
    probs = tf.sigmoid(logits).numpy().ravel()
    y_true.extend(y.numpy().astype(int))
    y_prob.extend(probs)

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# ==================================================
# CUSTOM METRICS
# ==================================================
def dice_score(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def mean_iou(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def mea(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# ==================================================
# METRICS
# ==================================================
y_true = np.array(y_true)
y_prob = np.array(y_prob)

# Convert probabilities to binary predictions
y_pred = (y_prob >= 0.5).astype(int)

# Now compute all metrics
acc  = accuracy_score(y_true, y_pred) * 100
prec = precision_score(y_true, y_pred) * 100
rec  = recall_score(y_true, y_pred) * 100
f1   = f1_score(y_true, y_pred) * 100
auc  = roc_auc_score(y_true, y_prob)
dice = dice_score(y_true, y_pred) * 100
miou = mean_iou(y_true, y_pred) * 100
mea_val = mea(y_true, y_pred) * 100


train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]

params = model.count_params() / 1e6
gflops = params * 2  # standard approximation

# ==================================================
# FINAL OUTPUT
# ==================================================
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

print("\n📊 ImTranNet-TriCore Evaluation Metrics of Chest X-ray Dataset")
print("======================================")
print(f"Accuracy (%)         : {acc:.2f}")
print(f"Precision (%)        : {prec:.2f}")
print(f"Recall (%)           : {rec:.2f}")
print(f"F1-Score (%)         : {f1:.2f}")
print(f"AUC                  : {auc:.4f}")
print(f"Dice Score (%)       : {dice:.2f}")
print(f"mIoU (%)             : {miou:.2f}")
print(f"MAE (%)              : {mea_val:.2f}")
print("--------------------------------------")
print(f"Training Loss        : {train_loss:.4f}")
print(f"Validation Loss      : {val_loss:.4f}")
print("--------------------------------------")
print(f"Parameters (M)       : {params:.2f}")
print(f"FLOPs (G)            : {gflops:.2f}")


Found 98 files belonging to 2 classes.
Found 98 files belonging to 2 classes.
Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 31s 16s/step - binary_accuracy: 0.4375 - loss: 0.7684 - val_binary_accuracy: 0.4898 - val_loss: 0.6873
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 21s 16s/step - binary_accuracy: 0.4583 - loss: 0.7302 - val_binary_accuracy: 0.4898 - val_loss: 0.6788
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 20s 16s/step - binary_accuracy: 0.4583 - loss: 0.7451 - val_binary_accuracy: 0.9286 - val_loss: 0.6720
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 12s 11s/step - binary_accuracy: 0.5000 - loss: 0.8263 - val_binary_accuracy: 0.9286 - val_loss: 0.6687
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 29s 25s/step - binary_accuracy: 0.3958 - loss: 0.8093 - val_binary_accuracy: 0.9286 - val_loss: 0.6624
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 32s 28s/step - binary_accuracy: 0.6250 - loss: 0.6980 - val_binary_accuracy: 0.7449 - val_loss: 0.6578
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 30s 25s/step - binary_accuracy: 0.6250 - loss: 0.654

ablation

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ==================================================
# CORE-1: LM-AdaFilter (FIXED residual mismatch)
# ==================================================
class LMAdaFilter(layers.Layer):
    def __init__(self, out_channels=16):
        super().__init__()
        self.out_channels = out_channels

        self.dw3 = layers.DepthwiseConv2D(3, padding="same")
        self.dw5 = layers.DepthwiseConv2D(5, padding="same")

        self.w = self.add_weight(
            shape=(2,),
            initializer="ones",
            trainable=True,
            name="scale_weights"
        )

        self.pw = layers.Conv2D(out_channels, 1, padding="same")
        self.norm = layers.LayerNormalization()
        self.act = layers.ReLU()
        self.shortcut = None

    def build(self, input_shape):
        in_channels = input_shape[-1]
        if in_channels != self.out_channels:
            self.shortcut = layers.Conv2D(self.out_channels, 1, padding="same")
        else:
            self.shortcut = lambda x: x
        super().build(input_shape)

    def call(self, x):
        f1 = self.dw3(x)
        f2 = self.dw5(x)

        w = tf.nn.softmax(self.w)
        out = w[0] * f1 + w[1] * f2

        out = self.pw(out)
        out = self.norm(out)

        return self.act(out + self.shortcut(x))


# ==================================================
# CORE-2: DP-AtRes-SRU (FAST, SAFE VERSION)
# ==================================================
class DP_AtRes_SRU_FAST(layers.Layer):
    def __init__(self, channels=32):
        super().__init__()
        self.conv1 = layers.Conv2D(channels, 3, padding="same")
        self.conv2 = layers.Conv2D(channels, 3, padding="same")
        self.norm = layers.BatchNormalization()
        self.act = layers.ReLU()

    def call(self, x):
        residual = x
        x = self.act(self.norm(self.conv1(x)))
        x = self.norm(self.conv2(x))
        return self.act(x + residual)


# ==================================================
# CORE-3: MHHT (simplified & stable)
# ==================================================
class MHHT(layers.Layer):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )
        self.norm = layers.LayerNormalization()

    def call(self, x):
        attn_out = self.attn(x, x)
        return self.norm(x + attn_out)


# ==================================================
# SHARED COMPILE FUNCTION
# ==================================================
def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["binary_accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model


# ==================================================
# A1: LMAdaFilter ONLY
# ==================================================
def build_LMAdaFilter_only(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = LMAdaFilter()(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return compile_model(models.Model(inputs, outputs, name="A1_LMAdaFilter"))


# ==================================================
# A2: DP-AtRes-SRU ONLY
# ==================================================
def build_DP_AtRes_SRU_only(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same")(inputs)
    x = DP_AtRes_SRU_FAST(32)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return compile_model(models.Model(inputs, outputs, name="A2_DP_AtRes_SRU"))


# ==================================================
# A3: MHHT ONLY
# ==================================================
def build_MHHT_only(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 1)(inputs)
    x = layers.Reshape((-1, 32))(x)  # (H*W, C)
    x = MHHT(embed_dim=32)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return compile_model(models.Model(inputs, outputs, name="A3_MHHT"))


# ==================================================
# A4: LMAdaFilter + DP-AtRes-SRU
# ==================================================
def build_LMAda_DP_AtRes(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = LMAdaFilter()(inputs)
    x = DP_AtRes_SRU_FAST(16)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return compile_model(models.Model(inputs, outputs, name="A4_LMAda_DP_AtRes"))

# ==================================================
# RUN ABLATION STUDY
# ==================================================
input_shape = (50, 50, 3)

models_dict = {
    "A1_LMAdaFilter": build_LMAdaFilter_only(input_shape),
    "A2_DP_AtRes_SRU": build_DP_AtRes_SRU_only(input_shape),
    "A3_MHHT": build_MHHT_only(input_shape),
    "A4_LMAda_DP_AtRes": build_LMAda_DP_AtRes(input_shape),
}

histories = {}

results = {}

for name, model in models_dict.items():
    print(f"\n🚀 Training {name}")
    model.summary()

    model.fit(
        train_ds,
        validation_data=test_ds,
        epochs=10,
        verbose=1,
        steps_per_epoch=2
    )

    # Evaluate accuracy for this model
    loss, acc, auc = model.evaluate(test_ds, verbose=0)
    acc_percent = acc * 100

    results[name] = acc_percent
    print(f"{name} Accuracy of Chest X-ray Dataset: {acc_percent:.2f}%")





🚀 Training A1_LMAdaFilter


Model: "A1_LMAdaFilter"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lm_ada_filter_4 (LMAdaFilter)   │ (None, 50, 50, 16)     │           270 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 16)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,423 (5.56 KB)

 Trainable params: 1,423 (5.56 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 50, 50, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lm_ada_filter_5 (LMAdaFilter)   │ (None, 50, 50, 16)     │           270 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dp__at_res_sru_fast_5           │ (None, 50, 50, 16)     │         4,704 │
│ (DP_AtRes_SRU_FAST)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 16)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,127 (23.93 KB)

 Trainable params: 6,095 (23.81 KB)

 Non-trainable params: 32 (128.00 B)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - auc: 0.8655 - binary_accuracy: 0.5000 - loss: 0.6688 - val_auc: 0.7808 - val_binary_accuracy: 0.5102 - val_loss: 0.6906
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 687ms/step - auc: 0.9656 - binary_accuracy: 0.5625 - loss: 0.6620 - val_auc: 0.8560 - val_binary_accuracy: 0.5102 - val_loss: 0.6897
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 733ms/step - auc: 0.9239 - binary_accuracy: 0.6458 - loss: 0.6539 - val_auc: 0.8929 - val_binary_accuracy: 0.5102 - val_loss: 0.6895
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 603ms/step - auc: 0.0000e+00 - binary_accuracy: 0.0000e+00 - loss: 0.7819 - val_auc: 0.9271 - val_binary_accuracy: 0.5102 - val_loss: 0.6897
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 815ms/step - auc: 0.8729 - binary_accuracy: 0.7917 - loss: 0.6523 - val_auc: 0.8977 - val_binary_accuracy: 0.5102 - val_loss: 0.6904
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step - auc: 0.8873 - binary_accuracy: 0.6250 - loss: 0.6466 - val_auc: 0.9048 - val_binary